# Trabalho Prático 05 — Agentes Inteligentes para Othello

**Disciplina:** GCC128 — Inteligência Artificial  
**Tecnologias:** Python 3, LangChain, LangGraph e Groq

O notebook implementa Othello, um agente heurístico e um agente LLM com dois modos de autonomia.

## 1. Configuração do ambiente

Na raiz do projeto, crie o ambiente virtual e instale as dependências:

**Windows (PowerShell):**
```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
```

**Ubuntu:**
```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

No VS Code, abra o notebook, clique em **Selecionar Kernel** e escolha o Python do ambiente `.venv`.


### 1.1 Verificar as dependências

A célula importa as bibliotecas e exibe suas versões.

In [188]:
import sys
from importlib.metadata import version

import pygame
import langchain
import langgraph
import langchain_groq

print("Python:", sys.version.split()[0])
for pacote in ("pygame", "langchain", "langgraph", "langchain-groq"):
    print(f"{pacote}: {version(pacote)}")
print("Ambiente configurado. Nenhum agente foi executado.")

Python: 3.12.3
pygame: 2.6.1
langchain: 1.5.1
langgraph: 1.2.22
langchain-groq: 1.2.0
Ambiente configurado. Nenhum agente foi executado.


## 2. Motor e estado da partida

O motor implementa as regras do tabuleiro 8×8. O estado guarda o tabuleiro, o jogador atual e o histórico.

### 2.1 Tabuleiro e representação

In [189]:
# Representação das casas
VAZIO = 0
PRETO = 1
BRANCO = -1

TAMANHO = 8

def criar_tabuleiro():
    tabuleiro = [
        [VAZIO for _ in range(TAMANHO)]
        for _ in range(TAMANHO)
    ]

    # Posições iniciais das peças
    tabuleiro[3][3] = BRANCO
    tabuleiro[3][4] = PRETO
    tabuleiro[4][3] = PRETO
    tabuleiro[4][4] = BRANCO

    return tabuleiro


def exibir_tabuleiro(tabuleiro):
    simbolos = {
        VAZIO: ".",
        PRETO: "P",
        BRANCO: "B"
    }

    print("    " + " ".join(str(i) for i in range(TAMANHO)))

    for i, linha in enumerate(tabuleiro):
        print(f"{i}   " + " ".join(simbolos[peca] for peca in linha))

### 2.2 Capturas e movimentos legais

In [190]:
# Oito direções possíveis: horizontal, vertical e diagonais
DIRECOES = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1)
]


def pecas_capturadas(tabuleiro, linha, coluna, jogador):
    if not (0 <= linha < TAMANHO and 0 <= coluna < TAMANHO):
        return []

    if tabuleiro[linha][coluna] != VAZIO:
        return []

    capturadas = []

    for dl, dc in DIRECOES:
        l, c = linha + dl, coluna + dc
        caminho = []

        # Percorre as peças adversárias nesta direção
        while (
            0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == -jogador
        ):
            caminho.append((l, c))
            l += dl
            c += dc

        # A sequência só é válida se terminar em uma peça própria
        if (
            caminho
            and 0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == jogador
        ):
            capturadas.extend(caminho)

    return capturadas


def movimentos_validos(tabuleiro, jogador):
    movimentos = []

    for linha in range(TAMANHO):
        for coluna in range(TAMANHO):
            if pecas_capturadas(tabuleiro, linha, coluna, jogador):
                movimentos.append((linha, coluna))

    return movimentos

### 2.3 Aplicação de jogadas

In [191]:
def fazer_jogada(tabuleiro, linha, coluna, jogador):
    # Identifica as peças que serão capturadas
    capturadas = pecas_capturadas(
        tabuleiro, linha, coluna, jogador
    )

    # Impede movimentos inválidos
    if not capturadas:
        return False

    # Coloca a peça do jogador
    tabuleiro[linha][coluna] = jogador

    # Inverte as peças adversárias capturadas
    for l, c in capturadas:
        tabuleiro[l][c] = jogador

    return True

### 2.4 Alternância e passagem de turno

In [192]:
def proximo_turno(tabuleiro, jogador_atual):
    adversario = -jogador_atual

    # Verifica se o adversário pode jogar
    if movimentos_validos(tabuleiro, adversario):
        return adversario

    # Se o adversário não puder, o jogador atual continua
    if movimentos_validos(tabuleiro, jogador_atual):
        return jogador_atual

    # Nenhum jogador possui movimentos válidos
    return None

### 2.5 Placar e encerramento

In [193]:
def contar_pecas(tabuleiro):
    pretas = sum(linha.count(PRETO) for linha in tabuleiro)
    brancas = sum(linha.count(BRANCO) for linha in tabuleiro)

    return pretas, brancas


def jogo_terminou(tabuleiro):
    return (
        not movimentos_validos(tabuleiro, PRETO)
        and not movimentos_validos(tabuleiro, BRANCO)
    )


def vencedor_por_placar(pretas, brancas):
    if pretas > brancas:
        return PRETO
    if brancas > pretas:
        return BRANCO
    return VAZIO  # Empate


def obter_vencedor(tabuleiro):
    if not jogo_terminou(tabuleiro):
        return None
    return vencedor_por_placar(*contar_pecas(tabuleiro))

### 2.6 Estado inicial e histórico

In [194]:
def iniciar_partida():
    return {
        "tabuleiro": criar_tabuleiro(),
        "jogador_atual": PRETO,
        "historico": []
    }


# Estado compartilhado entre as ferramentas e os agentes.
estado_jogo = iniciar_partida()

In [195]:
def obter_historico():
    historico = estado_jogo["historico"]

    if not historico:
        return "Nenhuma jogada realizada."

    jogadas = []
    for numero, jogada in enumerate(historico, start=1):
        jogador = "PRETO" if jogada["player"] == PRETO else "BRANCO"
        jogadas.append(
            f"{numero}. {jogador}: "
            f"linha {jogada['row']}, coluna {jogada['col']}"
        )

    return "\n".join(jogadas)

## 3. Algoritmos de estratégia

### 3.1 Heurística posicional

A avaliação soma os pesos das peças do jogador e subtrai os pesos das peças adversárias.

In [196]:
# Pesos estratégicos das posições do tabuleiro
PESOS_POSICAO = [
    [100, -20,  10,   5,   5,  10, -20, 100],
    [-20, -50,  -2,  -2,  -2,  -2, -50, -20],
    [ 10,  -2,   5,   1,   1,   5,  -2,  10],
    [  5,  -2,   1,   0,   0,   1,  -2,   5],
    [  5,  -2,   1,   0,   0,   1,  -2,   5],
    [ 10,  -2,   5,   1,   1,   5,  -2,  10],
    [-20, -50,  -2,  -2,  -2,  -2, -50, -20],
    [100, -20,  10,   5,   5,  10, -20, 100]
]

In [197]:
def avaliar_tabuleiro(tabuleiro, jogador):
    pontuacao = 0

    for linha in range(TAMANHO):
        for coluna in range(TAMANHO):

            peca = tabuleiro[linha][coluna]

            if peca == jogador:
                pontuacao += PESOS_POSICAO[linha][coluna]

            elif peca == -jogador:
                pontuacao -= PESOS_POSICAO[linha][coluna]

    return pontuacao

### 3.2 Simulação e escolha da jogada

A heurística avalia as jogadas legais em cópias do tabuleiro. Em caso de empate, escolhe a primeira na ordem do motor.

In [198]:
def simular_jogada(tabuleiro, linha, coluna, jogador):
    simulacao = [fileira.copy() for fileira in tabuleiro]
    fazer_jogada(simulacao, linha, coluna, jogador)
    return simulacao


def avaliar_jogadas(tabuleiro, jogador):
    resultados = []

    for linha, coluna in movimentos_validos(tabuleiro, jogador):

        simulacao = simular_jogada(tabuleiro, linha, coluna, jogador)
        pontuacao = avaliar_tabuleiro(simulacao, jogador)

        resultados.append({
            "row": linha,
            "col": coluna,
            "score": pontuacao
        })

    return resultados

In [199]:
def escolher_melhor_jogada(tabuleiro, jogador):
    jogadas = avaliar_jogadas(tabuleiro, jogador)

    if not jogadas:
        return None

    melhor_jogada = jogadas[0]

    for jogada in jogadas:
        if jogada["score"] > melhor_jogada["score"]:
            melhor_jogada = jogada

    return melhor_jogada

### 3.3 Planejamento de respostas

O planejamento compara a pontuação imediata de cada jogada com a menor pontuação após uma resposta adversária. Quando o adversário passa a vez, mantém a pontuação imediata.

In [200]:
def planejar_jogadas(tabuleiro, jogador):
    analises = []

    for linha, coluna in movimentos_validos(tabuleiro, jogador):
        simulacao = simular_jogada(tabuleiro, linha, coluna, jogador)

        pontuacao_atual = avaliar_tabuleiro(simulacao, jogador)
        respostas = movimentos_validos(simulacao, -jogador)
        pior_pontuacao = pontuacao_atual

        if respostas:
            pontuacoes = []
            for linha_resposta, coluna_resposta in respostas:
                resposta = simular_jogada(
                    simulacao, linha_resposta, coluna_resposta, -jogador
                )
                pontuacoes.append(avaliar_tabuleiro(resposta, jogador))

            pior_pontuacao = min(pontuacoes)

        analises.append({
            "row": linha,
            "col": coluna,
            "score_now": pontuacao_atual,
            "worst_response": pior_pontuacao,
            "opponent_moves": len(respostas)
        })

    return analises

### 3.4 Algoritmo Minimax

Avalia jogadas futuras considerando que ambos os jogadores
buscam obter o melhor resultado possível.

In [201]:
def minimax(tabuleiro, jogador_atual, jogador_raiz, profundidade,
            alpha=float("-inf"), beta=float("inf"), usar_poda=True):

    movimentos = movimentos_validos(tabuleiro, jogador_atual)

    if not movimentos:
        if not movimentos_validos(tabuleiro, -jogador_atual):
            pretas, brancas = contar_pecas(tabuleiro)
            diferenca = (pretas - brancas) * jogador_raiz

            if diferenca > 0:
                return 10000 + diferenca
            if diferenca < 0:
                return -10000 + diferenca
            return 0

        # Passagem de vez
        return minimax(
            tabuleiro, -jogador_atual, jogador_raiz,
            profundidade, alpha, beta, usar_poda
        )

    if profundidade <= 0:
        return avaliar_tabuleiro(tabuleiro, jogador_raiz)

    maximizando = jogador_atual == jogador_raiz
    melhor = float("-inf") if maximizando else float("inf")

    for linha, coluna in movimentos:
        simulacao = [l.copy() for l in tabuleiro]
        fazer_jogada(simulacao, linha, coluna, jogador_atual)

        pontuacao = minimax(
            simulacao, -jogador_atual, jogador_raiz,
            profundidade - 1, alpha, beta, usar_poda
        )

        if maximizando:
            melhor = max(melhor, pontuacao)
            alpha = max(alpha, melhor)
        else:
            melhor = min(melhor, pontuacao)
            beta = min(beta, melhor)

        # Poda alfa-beta
        if usar_poda and alpha >= beta:
            break

    return melhor

In [202]:
def escolher_jogada_minimax(tabuleiro, jogador,
                            profundidade=3, usar_poda=True):

    melhor_jogada = None
    melhor_pontuacao = float("-inf")

    for linha, coluna in movimentos_validos(tabuleiro, jogador):
        simulacao = [l.copy() for l in tabuleiro]
        fazer_jogada(simulacao, linha, coluna, jogador)

        pontuacao = minimax(
            simulacao,
            -jogador,
            jogador,
            profundidade - 1,
            alpha=melhor_pontuacao,
            usar_poda=usar_poda
        )

        if pontuacao > melhor_pontuacao:
            melhor_pontuacao = pontuacao
            melhor_jogada = {
                "row": linha,
                "col": coluna,
                "score": pontuacao
            }

    return melhor_jogada

## 4. Ferramentas do jogo

As oito ferramentas operam sobre o estado compartilhado e usam as regras e os cálculos das seções anteriores. As coordenadas começam em zero.

### 4.1 Tabuleiro, movimentos legais e situação da partida

In [203]:
from langchain_core.tools import tool


@tool
def get_board_state() -> dict:
    """Consulta o estado atual do tabuleiro de Othello."""

    tabuleiro = estado_jogo["tabuleiro"]
    pretas, brancas = contar_pecas(tabuleiro)

    return {
        "board": [linha.copy() for linha in tabuleiro],
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas
    }


@tool
def get_valid_moves() -> dict:
    """Consulta os movimentos válidos do jogador atual.
    As coordenadas de linha e coluna começam em zero.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {"current_player": None, "moves": []}

    movimentos = movimentos_validos(
        estado_jogo["tabuleiro"], jogador
    )

    return {
        "current_player": jogador,
        "moves": [
            {"row": linha, "col": coluna}
            for linha, coluna in movimentos
        ]
    }

In [204]:
@tool
def get_game_status() -> dict:
    """Consulta o estado da partida de Othello, incluindo
    pontuação, jogador atual, encerramento e vencedor.
    """

    tabuleiro = estado_jogo["tabuleiro"]

    pretas, brancas = contar_pecas(tabuleiro)
    terminou = jogo_terminou(tabuleiro)

    return {
        "game_over": terminou,
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas,
        "total_moves": len(estado_jogo["historico"]),
        "winner": vencedor_por_placar(pretas, brancas) if terminou else None
    }

### 4.2 Recomendação, histórico e planejamento

In [205]:
@tool
def get_heuristic_move() -> dict:
    """Recomenda a melhor jogada pela heurística posicional,
    sem executar o movimento.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {
            "current_player": None,
            "recommended_move": None
        }

    recomendacao = escolher_melhor_jogada(
        estado_jogo["tabuleiro"], jogador
    )

    return {
        "current_player": jogador,
        "recommended_move": recomendacao
    }

In [206]:
@tool
def get_move_history() -> dict:
    """Consulta o histórico de jogadas realizadas na partida."""

    historico = estado_jogo["historico"]

    return {
        "total_moves": len(historico),
        "moves": [
            {
                "player": jogada["player"],
                "row": jogada["row"],
                "col": jogada["col"]
            }
            for jogada in historico
        ]
    }

In [207]:
@tool
def get_strategic_plan() -> dict:
    """Avalia as jogadas válidas considerando a pontuação
    posicional e a pior resposta possível do adversário.
    Não executa nenhuma jogada.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {"current_player": None, "planning": []}

    return {
        "current_player": jogador,
        "planning": planejar_jogadas(
            estado_jogo["tabuleiro"], jogador
        )
    }

### 4.3 Execução de jogadas

`make_move` aplica uma jogada legal, registra suas coordenadas e atualiza o turno.

In [208]:
@tool
def make_move(row: int, col: int) -> dict:
    """Executa uma jogada de Othello para o jogador atual.
    As coordenadas row e col devem estar entre 0 e 7.
    """

    tabuleiro = estado_jogo["tabuleiro"]
    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {
            "success": False,
            "message": "A partida já terminou."
        }

    if not fazer_jogada(tabuleiro, row, col, jogador):
        return {
            "success": False,
            "message": "Jogada inválida.",
            "valid_moves": movimentos_validos(tabuleiro, jogador)
        }

    estado_jogo["historico"].append({
        "player": jogador,
        "row": row,
        "col": col
    })

    proximo = proximo_turno(tabuleiro, jogador)
    estado_jogo["jogador_atual"] = proximo

    return {
        "success": True,
        "message": "Jogada realizada com sucesso.",
        "next_player": proximo,
        "game_over": proximo is None
    }

### 4.4 Recomendação por Minimax

In [209]:
@tool
def get_minimax_move() -> dict:
    """Recomenda uma jogada utilizando Minimax com
    profundidade 3, sem modificar o tabuleiro.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {
            "current_player": None,
            "recommended_move": None
        }

    return {
        "current_player": jogador,
        "depth": 3,
        "recommended_move": escolher_jogada_minimax(
            estado_jogo["tabuleiro"], jogador
        )
    }

### 4.5 Registro das oito ferramentas

O modo estratégico usa quatro consultas fixas e oferece três consultas complementares à LLM. O modo operacional oferece as oito ferramentas à LLM desde o início do turno.

In [210]:
ferramentas = {
    "get_board_state": get_board_state,
    "get_valid_moves": get_valid_moves,
    "get_game_status": get_game_status,
    "get_heuristic_move": get_heuristic_move,
    "get_move_history": get_move_history,
    "get_strategic_plan": get_strategic_plan,
    "make_move": make_move,
    "get_minimax_move": get_minimax_move,
}

## 5. Integração com Groq e LangGraph

### 5.1 Credenciais e modelo

A célula solicita a chave Groq se `GROQ_API_KEY` estiver ausente do ambiente.

Defina `MODELO_LLM` na célula do modelo. O padrão é `openai/gpt-oss-20b`.

In [211]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")

In [ ]:
from langchain_groq import ChatGroq
from pydantic import BaseModel, ConfigDict, Field

MODELO_LLM = "openai/gpt-oss-20b"

llm = ChatGroq(
    model=MODELO_LLM,
    temperature=0
)

class DecisaoOthello(BaseModel):
    model_config = ConfigDict(extra="forbid")

    row: int = Field(description="Linha da jogada, entre 0 e 7")
    col: int = Field(description="Coluna da jogada, entre 0 e 7")
    justificativa: str = Field(description="Motivo estratégico resumido")

### 5.2 Modo estratégico

O grafo consulta tabuleiro, jogadas legais, histórico e Minimax. A LLM pode decidir na primeira resposta ou pedir até duas consultas diferentes entre heurística, planejamento e situação da partida. Ela devolve a decisão em JSON e o grafo valida e executa a jogada.

O contexto inclui o total de jogadas e as oito mais recentes. O histórico completo continua disponível na ferramenta original.

In [ ]:
import json
from time import perf_counter

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, ToolMessage
)


class EstadoAgente(MessagesState, total=False):
    jogador_turno: int
    historico_inicial: int
    jogadas_validas: list
    decisao: dict
    origem_decisao: str
    erro_decisao: str
    jogada_escolhida: dict
    resultado: dict
    ferramentas_automaticas: list
    metricas_chamadas_llm: list


ORIENTACOES_ESTRATEGICAS = """No início, priorize mobilidade, jogadas legais e posicionamento.
A partir de cerca de dez jogadas, considere consultar get_game_status e o histórico
quando ajudarem a decidir. No meio, considere Minimax, planejamento, controle
dos cantos e respostas adversárias. No final, priorize o resultado, a contagem
de peças e a previsão das últimas jogadas.
Decida assim que tiver informação suficiente. Reaproveite os dados recebidos.
Evite repetir consultas ou pedir dados equivalentes, especialmente no início."""


CAMPOS_TOKENS = {
    "tokens_entrada": ("input_tokens", "prompt_tokens"),
    "tokens_saida": ("output_tokens", "completion_tokens"),
    "tokens_total": ("total_tokens", "total_tokens")
}


def consultar_llm(modelo, mensagens, medicao):
    inicio = perf_counter()
    try:
        resposta = modelo.invoke(mensagens)
    finally:
        medicao["tempo_s"] = perf_counter() - inicio

    uso = resposta.usage_metadata or {}
    uso_groq = resposta.response_metadata.get("token_usage") or {}
    for campo, (chave, chave_groq) in CAMPOS_TOKENS.items():
        valor = uso.get(chave)
        medicao[campo] = uso_groq.get(chave_groq) if valor is None else valor
    return resposta


def iniciar_turno():
    jogador = estado_jogo["jogador_atual"]
    if jogador not in (PRETO, BRANCO):
        raise RuntimeError("Não há um turno válido para o agente.")
    return {
        "jogador_turno": jogador,
        "historico_inicial": len(estado_jogo["historico"])
    }


def erro_chamada(resposta, permitidas):
    chamadas = resposta.tool_calls
    if resposta.invalid_tool_calls:
        return "chamada_de_ferramenta_invalida"
    if len(chamadas) != 1:
        return "multiplas_chamadas_de_ferramenta" if chamadas else "resposta_sem_chamada_de_ferramenta"
    if chamadas[0]["name"] not in permitidas:
        return "ferramenta_desconhecida"
    return ""


def consultas_executadas(estado):
    return [
        mensagem.name for mensagem in estado["messages"]
        if isinstance(mensagem, ToolMessage) and mensagem.name
    ]


def validar_decisao(estado: EstadoAgente) -> dict:
    tabuleiro = estado_jogo["tabuleiro"]
    jogador = estado_jogo["jogador_atual"]
    legais = estado.get("jogadas_validas")
    if legais is None:
        legais = [
            {"row": linha, "col": coluna}
            for linha, coluna in movimentos_validos(tabuleiro, jogador)
        ]
    if not legais:
        raise RuntimeError("O jogador atual não possui jogadas válidas.")
    decisao = estado.get("decisao", {})
    escolhida = next(
        (
            movimento for movimento in legais
            if movimento["row"] == decisao.get("row")
            and movimento["col"] == decisao.get("col")
        ),
        None
    )

    if escolhida is None:
        planejamento = planejar_jogadas(tabuleiro, jogador)
        if planejamento:
            melhor = max(
                planejamento,
                key=lambda item: (
                    item["worst_response"],
                    item["score_now"],
                    -item["row"],
                    -item["col"]
                )
            )
            escolhida = {"row": melhor["row"], "col": melhor["col"]}
        else:
            escolhida = legais[0]

        return {
            "jogada_escolhida": escolhida,
            "origem_decisao": "fallback_heuristico",
            "erro_decisao": estado.get("erro_decisao") or "jogada_invalida"
        }

    return {"jogada_escolhida": escolhida}


def executar_jogada(estado: EstadoAgente) -> dict:
    jogada = estado["jogada_escolhida"]
    jogador = estado["jogador_turno"]
    quantidade_anterior = estado["historico_inicial"]
    if (
        estado_jogo["jogador_atual"] != jogador
        or len(estado_jogo["historico"]) != quantidade_anterior
    ):
        raise RuntimeError("O estado da partida mudou durante o turno.")

    resultado = ferramentas["make_move"].invoke({
        "row": jogada["row"],
        "col": jogada["col"]
    })

    if not resultado["success"]:
        raise RuntimeError(resultado["message"])

    if len(estado_jogo["historico"]) != quantidade_anterior + 1:
        raise RuntimeError("O turno deve registrar exatamente uma jogada.")

    registrada = estado_jogo["historico"][-1]
    if (
        registrada["player"] != jogador
        or registrada["row"] != jogada["row"]
        or registrada["col"] != jogada["col"]
    ):
        raise RuntimeError("A jogada registrada não corresponde à decisão.")

    automaticas = estado.get("ferramentas_automaticas", []) + ["make_move"]
    return {"resultado": resultado, "ferramentas_automaticas": automaticas}




CONSULTAS_FIXAS = (
    "get_board_state", "get_valid_moves", "get_move_history", "get_minimax_move"
)
CONSULTAS_COMPLEMENTARES = (
    "get_heuristic_move", "get_strategic_plan", "get_game_status"
)
MAX_CONSULTAS_COMPLEMENTARES = 2
tools_estrategicas = [ferramentas[nome] for nome in CONSULTAS_COMPLEMENTARES]
llm_estrategica = llm.bind_tools(tools_estrategicas)
no_consultas_estrategicas = ToolNode(tools_estrategicas, handle_tool_errors=True)

INSTRUCOES_ESTRATEGICAS = ORIENTACOES_ESTRATEGICAS + """
Você joga Othello. Use o tabuleiro, as jogadas legais, o histórico e a
recomendação Minimax recebidos. Se bastarem, decida na primeira resposta.
Se precisar de mais dados, solicite uma única
ferramenta por resposta, até duas consultas diferentes no turno.
Assim que puder decidir, responda somente com um objeto JSON com os campos
row, col e justificativa. As coordenadas começam em zero. O grafo fará a jogada."""


def observar(estado: EstadoAgente) -> dict:
    turno = iniciar_turno()
    consultas = {
        nome: ferramentas[nome].invoke({}) for nome in CONSULTAS_FIXAS
    }
    tabuleiro = consultas["get_board_state"]
    movimentos = consultas["get_valid_moves"]
    historico = consultas["get_move_history"]
    if (
        tabuleiro["current_player"] != turno["jogador_turno"]
        or movimentos["current_player"] != turno["jogador_turno"]
        or not movimentos["moves"]
    ):
        raise RuntimeError("O estado da partida não corresponde a um turno válido.")
    contexto = {
        "jogador": turno["jogador_turno"],
        "tabuleiro": tabuleiro["board"],
        "jogadas_validas": movimentos["moves"],
        "total_jogadas": historico["total_moves"],
        "ultimas_jogadas": historico["moves"][-8:],
        "recomendacao_minimax": consultas["get_minimax_move"]["recommended_move"]
    }
    return {
        **turno,
        "jogadas_validas": movimentos["moves"],
        "ferramentas_automaticas": list(CONSULTAS_FIXAS),
        "messages": [
            SystemMessage(content=INSTRUCOES_ESTRATEGICAS),
            HumanMessage(content=json.dumps(contexto, ensure_ascii=False, separators=(",", ":")))
        ]
    }


def decidir_estrategico(estado: EstadoAgente) -> dict:
    usadas = consultas_executadas(estado)
    disponiveis = [nome for nome in CONSULTAS_COMPLEMENTARES if nome not in usadas]
    pode_consultar = len(usadas) < MAX_CONSULTAS_COMPLEMENTARES
    medicao = {}
    atualizacao = {
        "metricas_chamadas_llm": estado.get("metricas_chamadas_llm", []) + [medicao]
    }
    try:
        resposta = consultar_llm(llm_estrategica, estado["messages"], medicao)
    except Exception as erro:
        return {**atualizacao, "erro_decisao": type(erro).__name__}

    if resposta.tool_calls or resposta.invalid_tool_calls:
        erro = erro_chamada(resposta, CONSULTAS_COMPLEMENTARES)
        if not erro and not pode_consultar:
            erro = "limite_de_consultas"
        elif not erro and resposta.tool_calls[0]["name"] not in disponiveis:
            erro = "consulta_repetida"
        return {**atualizacao, "messages": [resposta], "erro_decisao": erro}

    try:
        decisao = DecisaoOthello.model_validate_json(resposta.content)
    except (TypeError, ValueError):
        return {**atualizacao, "messages": [resposta], "erro_decisao": "decisao_invalida"}

    return {**atualizacao, "messages": [resposta], "decisao": decisao.model_dump(), "origem_decisao": "llm"}


def rotear_estrategico(estado: EstadoAgente) -> str:
    if estado.get("decisao") or estado.get("erro_decisao"):
        return "validar"
    return "consultar"


grafo_estrategico = StateGraph(EstadoAgente)
grafo_estrategico.add_node("observar", observar)
grafo_estrategico.add_node("decidir", decidir_estrategico)
grafo_estrategico.add_node("consultar", no_consultas_estrategicas)
grafo_estrategico.add_node("validar", validar_decisao)
grafo_estrategico.add_node("executar", executar_jogada)

grafo_estrategico.add_edge(START, "observar")
grafo_estrategico.add_edge("observar", "decidir")
grafo_estrategico.add_conditional_edges(
    "decidir", rotear_estrategico, {"consultar": "consultar", "validar": "validar"}
)
grafo_estrategico.add_edge("consultar", "decidir")
grafo_estrategico.add_edge("validar", "executar")
grafo_estrategico.add_edge("executar", END)

app_estrategico = grafo_estrategico.compile()

### 5.3 Modo operacional

A LLM recebe o jogador e o número do turno. Pode escolher qualquer uma das oito ferramentas. O grafo aceita uma chamada por resposta e até seis por turno. Uma jogada válida encerra o turno. Quando a LLM falha ou esgota o limite, o grafo usa o mesmo fallback dos turnos estratégicos.

In [ ]:
INSTRUCOES_OPERACIONAIS = ORIENTACOES_ESTRATEGICAS + """
Você joga Othello. Escolha quais ferramentas consultar e em que ordem.
Se os dados já permitirem escolher uma jogada, chame make_move em seguida.
Faça uma única chamada por resposta. Você tem até seis chamadas e deve
reservar uma para make_move, que joga para o jogador atual. As coordenadas
começam em zero. Se make_move rejeitar uma jogada, use o retorno para corrigir
a escolha. Uma execução bem-sucedida encerra o turno."""


llm_com_ferramentas = llm.bind_tools(list(ferramentas.values()))
MAX_FERRAMENTAS_POR_TURNO = 6
no_ferramentas = ToolNode(list(ferramentas.values()), handle_tool_errors=True)


def preparar_operacional(estado: EstadoAgente) -> dict:
    turno = iniciar_turno()
    contexto = {
        "jogador": turno["jogador_turno"],
        "turno": turno["historico_inicial"] + 1
    }
    return {
        **turno,
        "messages": [
            SystemMessage(content=INSTRUCOES_OPERACIONAIS),
            HumanMessage(content=json.dumps(contexto, ensure_ascii=False, separators=(",", ":")))
        ]
    }


def decidir_operacional(estado: EstadoAgente) -> dict:
    medicao = {}
    atualizacao = {
        "metricas_chamadas_llm": estado.get("metricas_chamadas_llm", []) + [medicao]
    }
    try:
        resposta = consultar_llm(llm_com_ferramentas, estado["messages"], medicao)
    except Exception as erro:
        return {**atualizacao, "erro_decisao": type(erro).__name__}

    return {
        **atualizacao,
        "messages": [resposta],
        "erro_decisao": erro_chamada(resposta, ferramentas)
    }


def rotear_operacional(estado: EstadoAgente) -> str:
    if estado.get("erro_decisao"):
        return "validar"
    return "tools"


def rotear_apos_ferramenta(estado: EstadoAgente) -> str:
    if len(estado_jogo["historico"]) > estado["historico_inicial"]:
        return "concluir"
    if len(consultas_executadas(estado)) >= MAX_FERRAMENTAS_POR_TURNO:
        return "limite"
    return "decidir"


def atingir_limite(estado: EstadoAgente) -> dict:
    return {"erro_decisao": "limite_de_ferramentas"}


def concluir_operacional(estado: EstadoAgente) -> dict:
    return {"origem_decisao": "llm_tool_call"}


grafo_operacional = StateGraph(EstadoAgente)
grafo_operacional.add_node("preparar", preparar_operacional)
grafo_operacional.add_node("decidir", decidir_operacional)
grafo_operacional.add_node("tools", no_ferramentas)
grafo_operacional.add_node("limite", atingir_limite)
grafo_operacional.add_node("concluir", concluir_operacional)
grafo_operacional.add_node("validar", validar_decisao)
grafo_operacional.add_node("executar", executar_jogada)

grafo_operacional.add_edge(START, "preparar")
grafo_operacional.add_edge("preparar", "decidir")
grafo_operacional.add_conditional_edges(
    "decidir", rotear_operacional, {"tools": "tools", "validar": "validar"}
)
grafo_operacional.add_conditional_edges(
    "tools", rotear_apos_ferramenta,
    {"decidir": "decidir", "limite": "limite", "concluir": "concluir"}
)
grafo_operacional.add_edge("concluir", END)
grafo_operacional.add_edge("limite", "validar")
grafo_operacional.add_edge("validar", "executar")
grafo_operacional.add_edge("executar", END)

app_operacional = grafo_operacional.compile()

### 5.4 Visualização dos fluxos

Os diagramas Mermaid vêm dos grafos compilados. O estratégico mostra o ciclo de consultas complementares e a execução pelo grafo. O operacional mostra o ciclo de ferramentas, a saída após uma jogada válida e o caminho de fallback.

In [215]:
print("Modo estratégico:")
print(app_estrategico.get_graph().draw_mermaid())
print("\nModo operacional:")
print(app_operacional.get_graph().draw_mermaid())

Modo estratégico:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	observar(observar)
	decidir(decidir)
	consultar(consultar<hr/><small><em>__tools = get_heuristic_move, get_strategic_plan, get_game_status</em></small>)
	validar(validar)
	executar(executar)
	__end__([<p>__end__</p>]):::last
	__start__ --> observar;
	consultar --> decidir;
	decidir -.-> consultar;
	decidir -.-> validar;
	observar --> decidir;
	validar --> executar;
	executar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


Modo operacional:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	preparar(preparar)
	decidir(decidir)
	tools(tools<hr/><small><em>__tools = get_board_state, get_valid_moves, get_game_status, get_heuristic_move, get_move_history, get_strategic_plan, make_move, get_minimax_move</em></small>)
	limite(limite)
	concluir(concluir)
	validar(

### 5.5 Seleção do modo e execução de um turno

Escolha **1** para o modo estratégico ou **2** para o operacional. O histórico distingue as ferramentas automáticas do grafo, as solicitadas pela LLM e o total de ferramentas executadas.

Cada turno da LLM registra modelo, chamadas, tempos e tokens informados pela API. O resumo marca contagens indisponíveis ou parciais.

In [216]:
escolha = input(
    "Modo de autonomia (1 = estratégica, 2 = operacional): "
).strip()

if escolha not in ("1", "2"):
    raise ValueError("Escolha 1 (estratégica) ou 2 (operacional).")

MODO_AUTONOMIA = {
    "1": "estrategica",
    "2": "operacional"
}[escolha]

print("Modo selecionado:", MODO_AUTONOMIA)

Modo selecionado: estrategica


In [ ]:
def resumir_metricas(chamadas, tempo_turno):
    metricas = {
        "modelo": llm.model_name,
        "chamadas_llm": len(chamadas),
        "tempo_turno_s": tempo_turno,
        "tempo_llm_s": sum(chamada["tempo_s"] for chamada in chamadas),
        "chamadas_com_tokens": {}
    }
    for campo in CAMPOS_TOKENS:
        valores = [
            chamada[campo] for chamada in chamadas if chamada.get(campo) is not None
        ]
        metricas[campo] = sum(valores) if valores else None
        metricas["chamadas_com_tokens"][campo] = len(valores)
    return metricas


def formatar_tokens(metricas, campo):
    valor = metricas[campo]
    if valor is None:
        return "indisponível"
    if metricas["chamadas_com_tokens"][campo] < metricas["chamadas_llm"]:
        return f"{valor} (parcial)"
    return str(valor)


def executar_turno_agente():
    jogador = estado_jogo["jogador_atual"]
    if jogador is None:
        return None

    inicio_turno = perf_counter()
    quantidade_anterior = len(estado_jogo["historico"])
    app = app_estrategico if MODO_AUTONOMIA == "estrategica" else app_operacional
    resultado = app.invoke({"messages": []}, config={"recursion_limit": 20})

    if len(estado_jogo["historico"]) != quantidade_anterior + 1:
        raise RuntimeError("O agente deve executar exatamente uma jogada.")

    jogada = estado_jogo["historico"][-1]
    if jogada["player"] != jogador:
        raise RuntimeError("A jogada registrada pertence a outro jogador.")

    solicitadas = [
        chamada["name"]
        for mensagem in resultado["messages"]
        if isinstance(mensagem, AIMessage)
        for chamada in mensagem.tool_calls + mensagem.invalid_tool_calls
        if chamada.get("name")
    ]
    automaticas = resultado.get("ferramentas_automaticas", [])
    executadas_llm = consultas_executadas(resultado)
    if MODO_AUTONOMIA == "estrategica":
        executadas = automaticas[:-1] + executadas_llm + automaticas[-1:]
    else:
        executadas = executadas_llm + automaticas

    metricas = resumir_metricas(
        resultado.get("metricas_chamadas_llm", []), perf_counter() - inicio_turno
    )
    registro = {
        "metricas": metricas,
        "modo_autonomia": MODO_AUTONOMIA,
        "origem_decisao": resultado["origem_decisao"],
        "ferramentas_automaticas": automaticas,
        "ferramentas_solicitadas": solicitadas,
        "ferramentas_executadas": executadas
    }
    if resultado.get("erro_decisao"):
        registro["erro_decisao"] = resultado["erro_decisao"]

    jogada.update(registro)
    resultado.update(registro)
    print("Modo:", MODO_AUTONOMIA)
    print("Origem da decisão:", registro["origem_decisao"])
    print("Jogada:", (jogada["row"], jogada["col"]))
    print("Ferramentas automáticas:", automaticas)
    print("Ferramentas solicitadas:", solicitadas)
    print("Ferramentas executadas:", executadas)

    print(
        f"LLM {metricas['modelo']} | {metricas['chamadas_llm']} chamada(s) | "
        f"turno {metricas['tempo_turno_s']:.2f}s | LLM {metricas['tempo_llm_s']:.2f}s"
    )
    entrada, saida, total = (formatar_tokens(metricas, campo) for campo in CAMPOS_TOKENS)
    print(f"Tokens — entrada {entrada}, saída {saida}, total {total}")

    return resultado

## 6. Partidas e exemplos

`jogar_partida_heuristica()` coloca dois jogadores heurísticos em uma partida.

In [ ]:
def jogar_partida_heuristica():
    global estado_jogo
    estado_jogo = iniciar_partida()

    while estado_jogo["jogador_atual"] is not None:
        tabuleiro = estado_jogo["tabuleiro"]
        jogador = estado_jogo["jogador_atual"]
        jogada = escolher_melhor_jogada(tabuleiro, jogador)

        if jogada is None:
            raise ValueError("Jogador atual sem movimentos válidos.")

        resultado = make_move.invoke({
            "row": jogada["row"],
            "col": jogada["col"]
        })

        if not resultado["success"]:
            raise RuntimeError(resultado["message"])

    return get_game_status.invoke({})

### 6.1 Partida entre a LLM e o agente heurístico

`jogar_partida_completa()` usa o modo selecionado para as pretas e a heurística para as brancas. Retorna a situação final e o histórico.

In [ ]:
def jogar_partida_completa():
    global estado_jogo
    estado_jogo = iniciar_partida()

    while estado_jogo["jogador_atual"] is not None:
        jogador = estado_jogo["jogador_atual"]
        quantidade_anterior = len(estado_jogo["historico"])

        if jogador == PRETO:
            executar_turno_agente()
        else:
            jogada = escolher_melhor_jogada(
                estado_jogo["tabuleiro"], jogador
            )
            if jogada is None:
                raise ValueError("Jogador atual sem movimentos válidos.")

            resultado = make_move.invoke({
                "row": jogada["row"],
                "col": jogada["col"]
            })
            if not resultado["success"]:
                raise RuntimeError(resultado["message"])

            estado_jogo["historico"][-1].update({
                "modo_autonomia": "heuristica",
                "origem_decisao": "heuristica",
                "ferramentas_solicitadas": [],
                "ferramentas_executadas": ["make_move"]
            })

        if len(estado_jogo["historico"]) != quantidade_anterior + 1:
            raise RuntimeError("Cada turno deve executar uma jogada.")

    return {
        "status": get_game_status.invoke({}),
        "historico": [
            jogada.copy() for jogada in estado_jogo["historico"]
        ]
    }

### 6.2 Exemplos para execução manual

Consulte o plano do turno atual com `get_strategic_plan.invoke({})`.

Para iniciar uma partida nova e observar um turno da LLM, execute:

```python
estado_jogo = iniciar_partida()
resultado = executar_turno_agente()
print(estado_jogo["historico"][-1])
```

### 6.3 Comparação manual dos modos

Este exemplo prepara a mesma posição para os dois modos. Cada turno exibe suas métricas.

```python
from copy import deepcopy

estado_jogo = iniciar_partida()

# Simula 30 jogadas utilizando a heurística
for _ in range(30):
    jogada = escolher_melhor_jogada(
        estado_jogo["tabuleiro"],
        estado_jogo["jogador_atual"]
    )
    make_move.invoke({
        "row": jogada["row"],
        "col": jogada["col"]
    })

estado_base = deepcopy(estado_jogo)

# Testa os dois agentes na mesma posição
for modo in ["estrategica", "operacional"]:
    MODO_AUTONOMIA = modo
    estado_jogo = deepcopy(estado_base)

    executar_turno_agente()
```